# Animation performance debug notebook

This notebook helps diagnose jerky gu_toolkit animations. It reports the current timer backend, whether Plotly FigureWidget support is available, whether `anywidget` is missing, and basic render/animation timing metrics.

In [ ]:
%pip -q install plotly pandas scipy numpy sympy anywidget ipywidgets

In [ ]:
import import_setup
from pprint import pprint
import time
import numpy as np
import sympy as sp

from gu_toolkit import (
    Figure,
    get_default_animation_clock,
    plot,
    parameter,
    runtime_diagnostics,
    runtime_support_performance_snapshot,
)

In [ ]:
def report_environment():
    diagnostics = runtime_diagnostics()
    support = diagnostics.get("plotly_widget_support", {})
    print("Runtime diagnostics:")
    pprint(diagnostics)
    print()
    if not support.get("figurewidget_supported", False):
        print("WARNING: Plotly FigureWidget support is degraded.")
        print("Reason:", support.get("reason"))
        print("This usually means anywidget is missing in the current notebook kernel.")
        print("Interactive animation can look jerky when gu_toolkit must fall back to a plain plotly Figure.")
    else:
        print("Plotly FigureWidget support looks healthy.")
    return diagnostics


def figure_debug_summary(fig, *, recent_event_limit=8, include_layout_events=False):
    print(fig.performance_report(
        recent_event_limit=recent_event_limit,
        include_layout_events=include_layout_events,
    ))
    return fig.performance_snapshot(
        recent_event_limit=recent_event_limit,
        include_layout_events=include_layout_events,
    )


def benchmark_parameter_updates(fig, symbol, *, values=None, mode="flush"):
    ref = fig.parameters[symbol]
    if values is None:
        values = np.linspace(float(ref.min), float(ref.max), 120)
    values = [float(v) for v in values]

    started = time.perf_counter()
    for value in values:
        ref.value = value
        if mode == "flush":
            fig.flush_render_queue()
        elif mode == "force":
            fig.render(reason="benchmark_force", trigger={"value": value}, force=True)
        elif mode == "queued":
            pass
        else:
            raise ValueError(f"Unknown mode: {mode}")

    if mode == "queued":
        fig.flush_render_queue()

    elapsed_s = time.perf_counter() - started
    per_update_ms = (elapsed_s / max(1, len(values))) * 1000.0
    result = {
        "mode": mode,
        "updates": len(values),
        "elapsed_s": elapsed_s,
        "per_update_ms": per_update_ms,
    }
    print(result)
    return result


def animation_debug_snapshot(fig, symbol, *, recent_event_limit=8):
    control = fig.parameters.widget(symbol)
    animation = getattr(control, "_animation", None)
    if animation is None or not hasattr(animation, "performance_snapshot"):
        print("No animation controller diagnostics available for", symbol)
        return None
    snapshot = animation.performance_snapshot(recent_event_limit=recent_event_limit)
    pprint(snapshot)
    return snapshot


def clock_debug_snapshot(*, recent_event_limit=8):
    clock = get_default_animation_clock()
    snapshot = clock.performance_snapshot(recent_event_limit=recent_event_limit)
    pprint(snapshot)
    return snapshot

In [ ]:
report_environment()

In [ ]:
x, t = sp.symbols("x t")

fig_wave_perf = Figure(x_range=(-4, 4), y_range=(-1.5, 1.5), show=False)
with fig_wave_perf:
    plot(sp.sin(3 * x - t), x, id="wave")
    parameter(t, min=0.0, max=12.0, value=0.0, step=0.02)

fig_wave_perf

## Inspect the live runtime and figure state

In [ ]:
figure_debug_summary(fig_wave_perf, recent_event_limit=6)

In [ ]:
clock_debug_snapshot(recent_event_limit=6)

In [ ]:
animation_debug_snapshot(fig_wave_perf, t, recent_event_limit=6)

## Benchmark manual parameter stepping

`mode="flush"` forces the queued render to complete after every value update. `mode="force"` requests an immediate synchronous render through the figure API.

In [ ]:
benchmark_parameter_updates(fig_wave_perf, t, mode="flush")

In [ ]:
benchmark_parameter_updates(fig_wave_perf, t, mode="force")

## Try the built-in animation controls

Start the slider animation in the figure, let it run for a few seconds, then inspect the diagnostics again.

In [ ]:
figure_debug_summary(fig_wave_perf, recent_event_limit=10)

In [ ]:
clock_debug_snapshot(recent_event_limit=10)

In [ ]:
animation_debug_snapshot(fig_wave_perf, t, recent_event_limit=10)

## Interpreting the output

- `plotly_widget_support.figurewidget_supported == False` is a strong sign that `anywidget` is missing or FigureWidget creation failed.
- `timer_backend` shows which scheduler path gu_toolkit is using: asyncio, Tornado, browser `setTimeout`, or a thread timer fallback.
- `Animation clock` lateness and spacing metrics help spot scheduling jitter.
- `Render scheduler` shows whether requests are being coalesced or dropped heavily.
- Plot metrics such as `evaluate_ms`, `trace_update_ms`, and `x_reuse_hits` help separate math cost from rendering cost.